# 24 — Building a Real Ultrasound Classification Pipeline

In the previous notebook, we connected the complete PyTorch workflow from data splitting to final test evaluation.

Now we will make that workflow look much more like a real ultrasound research project.

Instead of storing images directly inside Python lists, we will work with:

- Image files on disk
- A metadata CSV
- Patient IDs
- Study IDs
- Site information
- Device information
- Train / validation / test split manifests

We will also study:

- Training-only normalization
- Variable image sizes
- Class imbalance
- Weighted loss
- Weighted sampling
- Site/device-aware evaluation
- Harmonization experiments
- Transfer learning
- Patient-level prediction aggregation
- Reproducible output folders

## In this notebook, we will study:

1. Reading image metadata from CSV
2. Patient-level split manifests
3. File-path based custom datasets
4. Loading grayscale ultrasound images
5. Resizing and normalization
6. Training-only augmentation
7. Handling variable image sizes
8. Class imbalance
9. Weighted losses and sampling
10. Baseline CNN for ultrasound
11. Transfer-learning baseline
12. Site/device-aware evaluation
13. Harmonization experiments
14. Patient-level prediction aggregation
15. Saving reproducible experiment outputs
16. Preparing for a real research-quality ultrasound study

## Main Goal

The complete file-based workflow is:

$$
\boxed{
CSV
\rightarrow
Patient\ Split
\rightarrow
Image\ Dataset
\rightarrow
Training\ Preprocessing
\rightarrow
DataLoader
\rightarrow
Model
\rightarrow
Validation
\rightarrow
Final\ Test
}
$$

For multi-site ultrasound, we also want to ask:

$$
\boxed{
\text{Does the model learn pathology}
\quad
\text{or acquisition/device shortcuts?}
}
$$


In [ ]:
import copy
import json
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image, ImageOps

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from torchvision import (
    transforms,
    models
)

import torchvision.transforms.functional as TF

print("PyTorch:", torch.__version__)


# 1. What a Real Ultrasound Metadata Table Might Contain

A practical CSV may contain columns such as:

$$
\begin{array}{|c|c|}
\hline
\textbf{Column} & \textbf{Meaning} \\
\hline
image\_path & \text{Path to image file} \\
\hline
label & \text{Target class} \\
\hline
patient\_id & \text{Patient identifier} \\
\hline
study\_id & \text{Examination/study identifier} \\
\hline
site & \text{Hospital/acquisition site} \\
\hline
device & \text{Scanner/device} \\
\hline
view & \text{Ultrasound view} \\
\hline
\end{array}
$$

The exact schema depends on your project.

The most important principle is:

> **Keep the metadata needed for splitting and subgroup evaluation.**


# 2. Example CSV Schema

A CSV might look conceptually like:

```text
image_path,label,patient_id,study_id,site,device,view
images/P001_0.png,0,P001,S001,Site_A,Device_X,view_1
images/P001_1.png,0,P001,S001,Site_A,Device_X,view_1
images/P002_0.png,1,P002,S002,Site_B,Device_Y,view_2
...
```

Notice that multiple images can belong to the same patient.


# 3. Creating a Small File-Based Demo Dataset

This notebook should run even if you do not yet connect your real ultrasound dataset.

So we will create a small **demo dataset on disk** that behaves like a real file-based project.

The demo images will have:

- Different spatial sizes
- Three classes
- Multiple images per patient
- Two sites
- Three devices
- Device-specific intensity differences

This lets us demonstrate the full pipeline safely.


In [ ]:
DEMO_ROOT = Path(
    "demo_ultrasound_project"
)

IMAGE_DIR = (
    DEMO_ROOT
    / "images"
)

IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_PATH = (
    DEMO_ROOT
    / "metadata.csv"
)

OUTPUT_DIR = (
    DEMO_ROOT
    / "outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    DEMO_ROOT.resolve()
)


# 4. Synthetic Ultrasound-Like Image Generator

This is only a teaching generator.

It creates:

- Class-specific structure
- Speckle-like noise
- Device-specific gain/contrast
- Variable image sizes

The scientific lessons come from the pipeline, not from the realism of these images.


In [ ]:
def make_demo_ultrasound_array(
    class_index,
    height,
    width,
    device_name,
    generator
):
    image = torch.zeros(
        height,
        width
    )

    cy = height // 2
    cx = width // 2

    if class_index == 0:
        image[
            max(0, cy - 5):
            min(height, cy + 5),
            max(0, cx - width // 4):
            min(width, cx + width // 4)
        ] = 0.85

    elif class_index == 1:
        radius_y = max(
            5,
            height // 7
        )

        radius_x = max(
            5,
            width // 7
        )

        yy = torch.arange(
            height
        ).view(
            -1,
            1
        )

        xx = torch.arange(
            width
        ).view(
            1,
            -1
        )

        ellipse = (
            (
                (yy - cy)
                / radius_y
            ) ** 2
            +
            (
                (xx - cx)
                / radius_x
            ) ** 2
            <= 1.0
        )

        image[
            ellipse
        ] = 0.95

    elif class_index == 2:
        image[
            max(0, cy - height // 5):
            min(height, cy + height // 5),
            max(0, cx - 4):
            min(width, cx + 4)
        ] = 0.95

        image[
            max(0, cy - 4):
            min(height, cy + 4),
            max(0, cx - width // 5):
            min(width, cx + width // 5)
        ] = 0.95

    else:
        raise ValueError(
            "class_index must be 0, 1, or 2"
        )

    speckle = torch.randn(
        image.shape,
        generator=generator
    ) * 0.10

    image = (
        image
        + speckle
    ).clamp(
        0.0,
        1.0
    )

    if device_name == "Device_A":
        gain = 0.90
        offset = 0.03

    elif device_name == "Device_B":
        gain = 1.10
        offset = 0.00

    else:
        gain = 0.75
        offset = 0.08

    image = (
        image
        * gain
        + offset
    ).clamp(
        0.0,
        1.0
    )

    return image


# 5. Writing Demo Images to Disk


In [ ]:
def tensor_to_uint8_image(
    tensor
):
    tensor = tensor.clamp(
        0.0,
        1.0
    )

    array = (
        tensor
        * 255.0
    ).round().to(
        torch.uint8
    ).numpy()

    return Image.fromarray(
        array,
        mode="L"
    )


def create_demo_metadata(
    root,
    image_dir,
    metadata_path,
    num_patients=72,
    images_per_patient=3
):
    records = []

    sites = [
        "Site_A",
        "Site_B"
    ]

    devices = [
        "Device_A",
        "Device_B",
        "Device_C"
    ]

    for patient_number in range(
        num_patients
    ):
        patient_id = (
            f"P{patient_number:03d}"
        )

        study_id = (
            f"S{patient_number:03d}"
        )

        class_index = (
            patient_number
            % 3
        )

        site = sites[
            patient_number
            % len(sites)
        ]

        device = devices[
            patient_number
            % len(devices)
        ]

        generator = (
            torch.Generator()
            .manual_seed(
                10000
                + patient_number
            )
        )

        for image_index in range(
            images_per_patient
        ):
            height = int(
                torch.randint(
                    72,
                    111,
                    (1,),
                    generator=generator
                ).item()
            )

            width = int(
                torch.randint(
                    78,
                    121,
                    (1,),
                    generator=generator
                ).item()
            )

            image_tensor = (
                make_demo_ultrasound_array(
                    class_index,
                    height,
                    width,
                    device,
                    generator
                )
            )

            filename = (
                f"{patient_id}_"
                f"{image_index}.png"
            )

            image_path = (
                image_dir
                / filename
            )

            tensor_to_uint8_image(
                image_tensor
            ).save(
                image_path
            )

            records.append({
                "image_path":
                    str(
                        image_path
                    ),

                "label":
                    class_index,

                "patient_id":
                    patient_id,

                "study_id":
                    study_id,

                "site":
                    site,

                "device":
                    device,

                "view":
                    f"view_{image_index % 2}"
            })

    dataframe = pd.DataFrame(
        records
    )

    dataframe.to_csv(
        metadata_path,
        index=False
    )

    return dataframe


if not METADATA_PATH.exists():
    demo_df = create_demo_metadata(
        DEMO_ROOT,
        IMAGE_DIR,
        METADATA_PATH
    )

else:
    demo_df = pd.read_csv(
        METADATA_PATH
    )

print(
    "Rows:",
    len(demo_df)
)

print(
    demo_df.head()
)


# 6. Reading Image Metadata From CSV

For your real dataset, this becomes:

```python
metadata = pd.read_csv(
    "your_metadata.csv"
)
```

Then immediately inspect:

- Columns
- Number of rows
- Missing values
- Class counts
- Patient counts
- Site counts
- Device counts


In [ ]:
metadata = pd.read_csv(
    METADATA_PATH
)

print(
    metadata.columns.tolist()
)

print(
    "Images:",
    len(metadata)
)

print(
    "Patients:",
    metadata[
        "patient_id"
    ].nunique()
)


# 7. Validate Required Columns

Fail early if important metadata is missing.


In [ ]:
required_columns = {
    "image_path",
    "label",
    "patient_id",
    "study_id",
    "site",
    "device"
}

missing_columns = (
    required_columns
    - set(
        metadata.columns
    )
)

assert not missing_columns, (
    f"Missing columns: "
    f"{missing_columns}"
)

print(
    "Required columns verified."
)


# 8. Check Missing Values


In [ ]:
print(
    metadata[
        sorted(
            required_columns
        )
    ].isna().sum()
)


# 9. Verify Image Files Exist

A metadata CSV can silently contain broken paths.

Check before training.


In [ ]:
missing_paths = [
    path
    for path in metadata[
        "image_path"
    ]
    if not Path(
        path
    ).exists()
]

print(
    "Missing files:",
    len(
        missing_paths
    )
)


# 10. Inspect Class Distribution


In [ ]:
class_counts = (
    metadata[
        "label"
    ]
    .value_counts()
    .sort_index()
)

print(
    class_counts
)


# 11. Inspect Site and Device Distribution


In [ ]:
print(
    "Sites:"
)

print(
    metadata[
        "site"
    ].value_counts()
)

print()

print(
    "Devices:"
)

print(
    metadata[
        "device"
    ].value_counts()
)


# 12. Why Patient-Level Splits Matter

If one patient contributes multiple images:

$$
Image_1,\ Image_2,\ Image_3
$$

those images are correlated.

If one enters training and another enters validation, the model may benefit from patient-specific information.

Therefore:

$$
\boxed{
Split\ Patients
\ First
}
$$

then assign all of that patient's images to the same split.


# 13. Verify One Patient Has One Target Class

For this classification setup, we assume each patient has one class label.

Check that assumption explicitly.


In [ ]:
labels_per_patient = (
    metadata
    .groupby(
        "patient_id"
    )[
        "label"
    ]
    .nunique()
)

problem_patients = (
    labels_per_patient[
        labels_per_patient
        != 1
    ]
)

print(
    "Patients with multiple labels:",
    len(
        problem_patients
    )
)


# 14. Build a Patient-Level Table

We need one row per patient for stratified splitting.


In [ ]:
patient_table = (
    metadata
    .groupby(
        "patient_id",
        as_index=False
    )
    .agg({
        "label":
            "first",

        "site":
            "first",

        "device":
            "first"
    })
)

print(
    patient_table.head()
)

print(
    "Patients:",
    len(
        patient_table
    )
)


# 15. Stratified Patient Split

We will split patients separately within each class.

That helps preserve class balance while still keeping patients independent.

For real research:

- Save the split manifest
- Do not regenerate it every experiment


In [ ]:
def stratified_patient_split(
    patient_table,
    train_fraction=0.70,
    val_fraction=0.15,
    seed=42
):
    rng = random.Random(
        seed
    )

    train_ids = []
    val_ids = []
    test_ids = []

    for class_index in sorted(
        patient_table[
            "label"
        ].unique()
    ):
        class_patients = (
            patient_table[
                patient_table[
                    "label"
                ]
                == class_index
            ][
                "patient_id"
            ]
            .tolist()
        )

        rng.shuffle(
            class_patients
        )

        n = len(
            class_patients
        )

        n_train = int(
            train_fraction
            * n
        )

        n_val = int(
            val_fraction
            * n
        )

        train_ids.extend(
            class_patients[
                :n_train
            ]
        )

        val_ids.extend(
            class_patients[
                n_train:
                n_train + n_val
            ]
        )

        test_ids.extend(
            class_patients[
                n_train + n_val:
            ]
        )

    return (
        set(
            train_ids
        ),
        set(
            val_ids
        ),
        set(
            test_ids
        )
    )


train_patient_ids, val_patient_ids, test_patient_ids = (
    stratified_patient_split(
        patient_table,
        seed=42
    )
)

print(
    len(train_patient_ids),
    len(val_patient_ids),
    len(test_patient_ids)
)


# 16. Verify Patient Independence


In [ ]:
assert train_patient_ids.isdisjoint(
    val_patient_ids
)

assert train_patient_ids.isdisjoint(
    test_patient_ids
)

assert val_patient_ids.isdisjoint(
    test_patient_ids
)

print(
    "No patient overlap."
)


# 17. Create a Split Manifest

A split manifest is one of the most important reproducibility files in the project.


In [ ]:
SPLIT_PATH = (
    DEMO_ROOT
    / "patient_split_manifest.json"
)

split_manifest = {
    "seed":
        42,

    "train_patient_ids":
        sorted(
            train_patient_ids
        ),

    "val_patient_ids":
        sorted(
            val_patient_ids
        ),

    "test_patient_ids":
        sorted(
            test_patient_ids
        )
}

SPLIT_PATH.write_text(
    json.dumps(
        split_manifest,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Saved:",
    SPLIT_PATH
)


# 18. Attach Split Labels to Every Image


In [ ]:
def split_name(
    patient_id
):
    if patient_id in train_patient_ids:
        return "train"

    if patient_id in val_patient_ids:
        return "val"

    if patient_id in test_patient_ids:
        return "test"

    raise ValueError(
        f"Unknown patient: "
        f"{patient_id}"
    )

metadata[
    "split"
] = metadata[
    "patient_id"
].map(
    split_name
)

print(
    metadata[
        "split"
    ].value_counts()
)


# 19. Save Image-Level Manifest

This is useful because every image now has an explicit split assignment.


In [ ]:
IMAGE_MANIFEST_PATH = (
    DEMO_ROOT
    / "image_manifest_with_split.csv"
)

metadata.to_csv(
    IMAGE_MANIFEST_PATH,
    index=False
)

print(
    "Saved:",
    IMAGE_MANIFEST_PATH
)


# 20. Check Split Class Counts


In [ ]:
split_class_table = pd.crosstab(
    metadata[
        "split"
    ],
    metadata[
        "label"
    ]
)

print(
    split_class_table
)


# 21. Check Site/Device Balance Across Splits

A patient-level split can still create site/device imbalance.

Inspect it.


In [ ]:
print(
    pd.crosstab(
        metadata[
            "split"
        ],
        metadata[
            "site"
        ]
    )
)

print()

print(
    pd.crosstab(
        metadata[
            "split"
        ],
        metadata[
            "device"
        ]
    )
)


# 22. Loading Grayscale Ultrasound Images

A robust image dataset should:

1. Read the file path
2. Open the image
3. Convert explicitly to grayscale
4. Apply preprocessing
5. Return label
6. Preserve metadata index

Using:

```python
Image.open(path).convert("L")
```

ensures one grayscale channel.


# 23. Handling Variable Image Sizes

Real ultrasound images may have different:

- Heights
- Widths
- Aspect ratios

A DataLoader cannot stack variable-sized tensors into one ordinary batch.

We need a spatial standardization strategy.


# 24. Option A — Direct Resize

The simplest method:

```python
Resize((128, 128))
```

This guarantees equal tensor size.

But it may distort aspect ratio.

Useful as a baseline, but not always ideal.


# 25. Option B — Preserve Aspect Ratio and Pad

A better option for many tasks:

1. Resize the long side
2. Preserve aspect ratio
3. Pad the remaining dimension

This avoids geometric stretching.


In [ ]:
class ResizeWithPadding:
    def __init__(
        self,
        size,
        fill=0
    ):
        self.size = int(
            size
        )

        self.fill = fill

    def __call__(
        self,
        image
    ):
        width, height = (
            image.size
        )

        scale = min(
            self.size
            / width,
            self.size
            / height
        )

        new_width = max(
            1,
            int(
                round(
                    width
                    * scale
                )
            )
        )

        new_height = max(
            1,
            int(
                round(
                    height
                    * scale
                )
            )
        )

        image = TF.resize(
            image,
            [
                new_height,
                new_width
            ]
        )

        pad_width = (
            self.size
            - new_width
        )

        pad_height = (
            self.size
            - new_height
        )

        left = (
            pad_width
            // 2
        )

        right = (
            pad_width
            - left
        )

        top = (
            pad_height
            // 2
        )

        bottom = (
            pad_height
            - top
        )

        image = ImageOps.expand(
            image,
            border=(
                left,
                top,
                right,
                bottom
            ),
            fill=self.fill
        )

        return image


# 26. Test Variable-Size Handling


In [ ]:
sample_path = metadata.iloc[
    0
][
    "image_path"
]

sample_image = Image.open(
    sample_path
).convert(
    "L"
)

resize_pad = ResizeWithPadding(
    128
)

resized_image = resize_pad(
    sample_image
)

print(
    "Original size:",
    sample_image.size
)

print(
    "New size:",
    resized_image.size
)


# 27. Compute Training-Only Mean and Standard Deviation

For grayscale images we need one:

$$
\mu_{train}
$$

and one:

$$
\sigma_{train}
$$

We will compute them after resizing/padding but before normalization.

For large real datasets, use a streaming approach instead of loading everything into memory.


In [ ]:
train_df = metadata[
    metadata[
        "split"
    ]
    == "train"
].reset_index(
    drop=True
)

val_df = metadata[
    metadata[
        "split"
    ]
    == "val"
].reset_index(
    drop=True
)

test_df = metadata[
    metadata[
        "split"
    ]
    == "test"
].reset_index(
    drop=True
)

print(
    len(train_df),
    len(val_df),
    len(test_df)
)


In [ ]:
def compute_grayscale_mean_std(
    dataframe,
    resize_transform
):
    pixel_sum = 0.0
    pixel_squared_sum = 0.0
    pixel_count = 0

    to_tensor = (
        transforms.ToTensor()
    )

    for path in dataframe[
        "image_path"
    ]:
        image = Image.open(
            path
        ).convert(
            "L"
        )

        image = resize_transform(
            image
        )

        tensor = to_tensor(
            image
        )

        pixel_sum += (
            tensor.sum().item()
        )

        pixel_squared_sum += (
            tensor.pow(
                2
            ).sum().item()
        )

        pixel_count += (
            tensor.numel()
        )

    mean = (
        pixel_sum
        / pixel_count
    )

    variance = (
        pixel_squared_sum
        / pixel_count
        - mean ** 2
    )

    std = math.sqrt(
        max(
            variance,
            1e-12
        )
    )

    return mean, std


train_mean, train_std = (
    compute_grayscale_mean_std(
        train_df,
        resize_transform=(
            ResizeWithPadding(
                128
            )
        )
    )
)

print(
    "Training mean:",
    train_mean
)

print(
    "Training std:",
    train_std
)


# 28. Why Training-Only Statistics?

Wrong:

$$
\mu_{all\ data}
$$

Correct:

$$
\boxed{
\mu_{train},\sigma_{train}
}
$$

Then reuse them on validation/test.

Otherwise preprocessing learns information from evaluation data.


# 29. Training-Only Augmentation

Training augmentation may include clinically plausible:

- Small rotations
- Mild translation
- Small scale changes
- Mild intensity changes

Validation and test preprocessing should usually be deterministic.


# 30. Baseline Grayscale Transforms

For the demo task we use mild geometric augmentation.

For real ultrasound:

> Validate every augmentation against anatomy and acquisition conventions.


In [ ]:
baseline_train_transform = (
    transforms.Compose([
        ResizeWithPadding(
            128
        ),

        transforms.RandomAffine(
            degrees=7,
            translate=(
                0.04,
                0.04
            ),
            scale=(
                0.95,
                1.05
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                train_mean
            ],
            std=[
                train_std
            ]
        )
    ])
)

baseline_eval_transform = (
    transforms.Compose([
        ResizeWithPadding(
            128
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                train_mean
            ],
            std=[
                train_std
            ]
        )
    ])
)


# 31. File-Path Based Ultrasound Dataset

This dataset will return:

- Image tensor
- Target
- Local row index

The index lets us recover:

- Patient
- Site
- Device
- Study metadata

during evaluation.


In [ ]:
class UltrasoundCSVDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = (
            transform
        )

    def __len__(self):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):
        row = self.dataframe.iloc[
            index
        ]

        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "L"
        )

        if self.transform is not None:
            image = self.transform(
                image
            )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            target,
            index
        )


# 32. Create Baseline Datasets


In [ ]:
baseline_train_dataset = (
    UltrasoundCSVDataset(
        train_df,
        transform=(
            baseline_train_transform
        )
    )
)

baseline_val_dataset = (
    UltrasoundCSVDataset(
        val_df,
        transform=(
            baseline_eval_transform
        )
    )
)

baseline_test_dataset = (
    UltrasoundCSVDataset(
        test_df,
        transform=(
            baseline_eval_transform
        )
    )
)

print(
    len(
        baseline_train_dataset
    )
)


# 33. Inspect One Loaded Sample


In [ ]:
sample_tensor, sample_target, sample_index = (
    baseline_train_dataset[
        0
    ]
)

print(
    "Tensor:",
    sample_tensor.shape
)

print(
    "Target:",
    sample_target
)

print(
    "Index:",
    sample_index
)


# 34. Class Imbalance

Real medical datasets are often imbalanced.

Suppose:

$$
90\%
$$

negative,

$$
10\%
$$

positive.

A model may achieve high accuracy while performing poorly on the minority class.

Always inspect training counts.


In [ ]:
train_class_counts = (
    train_df[
        "label"
    ]
    .value_counts()
    .sort_index()
)

print(
    train_class_counts
)


# 35. Class Weights for the Loss

One common strategy is to weight each class inversely to its frequency.

A simple balanced formula is:

$$
\boxed{
w_c
=
\frac{
N
}{
C\cdot N_c
}
}
$$

where:

- $N$ = total training samples
- $C$ = number of classes
- $N_c$ = samples in class $c$


In [ ]:
num_classes = int(
    metadata[
        "label"
    ].nunique()
)

counts_tensor = torch.tensor([
    int(
        train_class_counts.get(
            class_index,
            0
        )
    )
    for class_index in range(
        num_classes
    )
],
dtype=torch.float32)

class_weights = (
    counts_tensor.sum()
    / (
        num_classes
        * counts_tensor.clamp_min(
            1.0
        )
    )
)

print(
    "Counts:",
    counts_tensor
)

print(
    "Class weights:",
    class_weights
)


# 36. Weighted `CrossEntropyLoss`

Use:

```python
nn.CrossEntropyLoss(
    weight=class_weights
)
```

The weight must be on the same device as the model outputs.


# 37. Weighted Sampling

A second strategy is:

> Sample minority-class examples more frequently.

PyTorch provides:

`WeightedRandomSampler`.


In [ ]:
sample_weights = torch.tensor([
    class_weights[
        int(label)
    ].item()
    for label in baseline_train_dataset[
        "dataframe"
    ][
        "label"
    ].tolist()
],
dtype=torch.double)

weighted_sampler = (
    WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(
            sample_weights
        ),
        replacement=True
    )
)

print(
    weighted_sampler
)


# 38. Weighted Loss vs Weighted Sampling

$$
\begin{array}{|c|c|}
\hline
\textbf{Weighted Loss} & \textbf{Weighted Sampling} \\
\hline
Changes\ loss\ importance & Changes\ sample\ frequency \\
\hline
All\ data\ order\ can\ stay\ natural & Minority\ appears\ more\ often \\
\hline
Easy\ to\ implement & Changes\ effective\ training\ distribution \\
\hline
\end{array}
$$

Do not automatically use both.

Compare strategies on validation data.


# 39. Build DataLoaders

We will create:

- Ordinary shuffled training loader
- Weighted-sampling training loader
- Deterministic validation/test loaders


In [ ]:
baseline_train_loader = DataLoader(
    baseline_train_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=0
)

weighted_train_loader = DataLoader(
    baseline_train_dataset,
    batch_size=24,
    sampler=weighted_sampler,
    num_workers=0
)

baseline_val_loader = DataLoader(
    baseline_val_dataset,
    batch_size=48,
    shuffle=False,
    num_workers=0
)

baseline_test_loader = DataLoader(
    baseline_test_dataset,
    batch_size=48,
    shuffle=False,
    num_workers=0
)


# 40. Baseline CNN for Ultrasound

A good first model should be:

- Small
- Easy to debug
- One-channel native
- Independent of exact spatial resolution after convolution


In [ ]:
class UltrasoundBaselineCNN(nn.Module):
    def __init__(
        self,
        num_classes
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU()
        )

        self.pool = (
            nn.AdaptiveAvgPool2d(
                1
            )
        )

        self.classifier = nn.Linear(
            64,
            num_classes
        )

    def forward(
        self,
        x
    ):
        x = self.features(
            x
        )

        x = self.pool(
            x
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.classifier(
            x
        )


# 41. Verify Baseline Shape


In [ ]:
baseline_model = (
    UltrasoundBaselineCNN(
        num_classes
    )
)

dummy = torch.randn(
    4,
    1,
    128,
    128
)

print(
    baseline_model(
        dummy
    ).shape
)


# 42. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)


# 43. Training Function


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, targets, _ in loader:
        images = images.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_correct += (
            (
                logits.argmax(
                    dim=1
                )
                == targets
            )
            .sum()
            .item()
        )

        total_samples += (
            batch_size
        )

    return (
        total_loss
        / total_samples,
        total_correct
        / total_samples
    )


# 44. Validation Function


In [ ]:
def evaluate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for images, targets, _ in loader:
            images = images.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            total_correct += (
                (
                    logits.argmax(
                        dim=1
                    )
                    == targets
                )
                .sum()
                .item()
            )

            total_samples += (
                batch_size
            )

    return (
        total_loss
        / total_samples,
        total_correct
        / total_samples
    )


# 45. Best-Checkpoint Training Loop


In [ ]:
def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs
):
    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(
        epochs
    ):
        train_loss, train_acc = (
            train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device
            )
        )

        val_loss, val_acc = (
            evaluate_one_epoch(
                model,
                val_loader,
                criterion,
                device
            )
        )

        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_acc"
        ].append(
            val_acc
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.3f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:.3f}"
        )

    model.load_state_dict(
        best_state
    )

    return (
        history,
        best_val_loss
    )


# 46. Train an Unweighted Baseline


In [ ]:
torch.manual_seed(
    42
)

unweighted_model = (
    UltrasoundBaselineCNN(
        num_classes
    )
    .to(
        device
    )
)

unweighted_criterion = (
    nn.CrossEntropyLoss()
)

unweighted_optimizer = (
    torch.optim.AdamW(
        unweighted_model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
)

unweighted_history, unweighted_best_loss = (
    fit_model(
        unweighted_model,
        baseline_train_loader,
        baseline_val_loader,
        unweighted_criterion,
        unweighted_optimizer,
        device,
        epochs=8
    )
)


# 47. Train a Weighted-Loss Baseline


In [ ]:
torch.manual_seed(
    42
)

weighted_loss_model = (
    UltrasoundBaselineCNN(
        num_classes
    )
    .to(
        device
    )
)

weighted_criterion = (
    nn.CrossEntropyLoss(
        weight=class_weights.to(
            device
        )
    )
)

weighted_optimizer = (
    torch.optim.AdamW(
        weighted_loss_model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
)

weighted_history, weighted_best_loss = (
    fit_model(
        weighted_loss_model,
        baseline_train_loader,
        baseline_val_loader,
        weighted_criterion,
        weighted_optimizer,
        device,
        epochs=8
    )
)


# 48. Train a Weighted-Sampling Baseline


In [ ]:
torch.manual_seed(
    42
)

weighted_sampler_model = (
    UltrasoundBaselineCNN(
        num_classes
    )
    .to(
        device
    )
)

sampler_criterion = (
    nn.CrossEntropyLoss()
)

sampler_optimizer = (
    torch.optim.AdamW(
        weighted_sampler_model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
)

sampler_history, sampler_best_loss = (
    fit_model(
        weighted_sampler_model,
        weighted_train_loader,
        baseline_val_loader,
        sampler_criterion,
        sampler_optimizer,
        device,
        epochs=8
    )
)


# 49. Compare Imbalance Strategies


In [ ]:
print(
    "Unweighted best val loss:",
    unweighted_best_loss
)

print(
    "Weighted loss best val loss:",
    weighted_best_loss
)

print(
    "Weighted sampler best val loss:",
    sampler_best_loss
)


# 50. Do Not Choose Imbalance Strategy From Training Accuracy

Choose using validation performance and task-appropriate metrics.

For rare disease detection, also inspect:

- Per-class recall
- Macro F1
- Sensitivity
- Specificity
- AUPRC

depending on the task.


# 51. Collect Predictions With Metadata


In [ ]:
def collect_predictions(
    model,
    dataset,
    loader,
    device
):
    model.eval()

    rows = []

    with torch.inference_mode():
        for images, targets, indices in loader:
            images = images.to(
                device
            )

            logits = model(
                images
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            ).cpu()

            predictions = logits.argmax(
                dim=1
            ).cpu()

            targets = targets.cpu()

            for batch_position in range(
                len(
                    indices
                )
            ):
                local_index = int(
                    indices[
                        batch_position
                    ].item()
                )

                metadata_row = (
                    dataset
                    .dataframe
                    .iloc[
                        local_index
                    ]
                )

                record = {
                    "image_path":
                        metadata_row[
                            "image_path"
                        ],

                    "patient_id":
                        metadata_row[
                            "patient_id"
                        ],

                    "study_id":
                        metadata_row[
                            "study_id"
                        ],

                    "site":
                        metadata_row[
                            "site"
                        ],

                    "device":
                        metadata_row[
                            "device"
                        ],

                    "true_label":
                        int(
                            targets[
                                batch_position
                            ].item()
                        ),

                    "predicted_label":
                        int(
                            predictions[
                                batch_position
                            ].item()
                        )
                }

                for class_index in range(
                    probabilities.shape[
                        1
                    ]
                ):
                    record[
                        f"prob_class_{class_index}"
                    ] = float(
                        probabilities[
                            batch_position,
                            class_index
                        ].item()
                    )

                rows.append(
                    record
                )

    return pd.DataFrame(
        rows
    )


# 52. Choose a Baseline Model for Further Analysis

For demonstration, choose the model with lowest validation loss.


In [ ]:
candidate_models = [
    {
        "name":
            "unweighted",

        "model":
            unweighted_model,

        "val_loss":
            unweighted_best_loss
    },
    {
        "name":
            "weighted_loss",

        "model":
            weighted_loss_model,

        "val_loss":
            weighted_best_loss
    },
    {
        "name":
            "weighted_sampler",

        "model":
            weighted_sampler_model,

        "val_loss":
            sampler_best_loss
    }
]

selected_baseline = min(
    candidate_models,
    key=lambda item:
        item[
            "val_loss"
        ]
)

print(
    "Selected:",
    selected_baseline[
        "name"
    ]
)


# 53. Validation Predictions


In [ ]:
val_predictions_df = (
    collect_predictions(
        selected_baseline[
            "model"
        ],
        baseline_val_dataset,
        baseline_val_loader,
        device
    )
)

print(
    val_predictions_df.head()
)


# 54. Confusion Matrix From Prediction Table


In [ ]:
def confusion_matrix_from_dataframe(
    dataframe,
    num_classes
):
    matrix = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.long
    )

    for _, row in dataframe.iterrows():
        matrix[
            int(
                row[
                    "true_label"
                ]
            ),
            int(
                row[
                    "predicted_label"
                ]
            )
        ] += 1

    return matrix


val_confusion = (
    confusion_matrix_from_dataframe(
        val_predictions_df,
        num_classes
    )
)

print(
    val_confusion
)


# 55. Per-Class Recall


In [ ]:
def per_class_recall(
    confusion
):
    recalls = {}

    for class_index in range(
        confusion.shape[0]
    ):
        true_count = (
            confusion[
                class_index
            ].sum().item()
        )

        correct = (
            confusion[
                class_index,
                class_index
            ].item()
        )

        recalls[
            class_index
        ] = (
            correct
            / true_count
            if true_count > 0
            else float(
                "nan"
            )
        )

    return recalls


print(
    per_class_recall(
        val_confusion
    )
)


# 56. Site-Aware Evaluation

Overall accuracy may hide site-specific failure.

Evaluate each site separately.


In [ ]:
def subgroup_accuracy(
    dataframe,
    column
):
    rows = []

    for subgroup, group in (
        dataframe.groupby(
            column
        )
    ):
        accuracy = (
            group[
                "true_label"
            ].to_numpy()
            ==
            group[
                "predicted_label"
            ].to_numpy()
        ).mean()

        rows.append({
            column:
                subgroup,

            "n":
                len(
                    group
                ),

            "accuracy":
                float(
                    accuracy
                )
        })

    return pd.DataFrame(
        rows
    )


site_metrics = subgroup_accuracy(
    val_predictions_df,
    "site"
)

print(
    site_metrics
)


# 57. Device-Aware Evaluation


In [ ]:
device_metrics = (
    subgroup_accuracy(
        val_predictions_df,
        "device"
    )
)

print(
    device_metrics
)


# 58. Why Site/Device Evaluation Matters

A model may learn:

$$
Device
\rightarrow
Label
$$

instead of:

$$
Anatomy
\rightarrow
Label
$$

if acquisition settings correlate with the target.

Always inspect whether:

- Class balance differs by device
- Image statistics differ by device
- Performance differs by device


# 59. Device-Wise Training Intensity Statistics

Let's inspect the demo data.


In [ ]:
def mean_intensity_for_paths(
    paths
):
    values = []

    for path in paths:
        image = (
            transforms.ToTensor()(
                Image.open(
                    path
                ).convert(
                    "L"
                )
            )
        )

        values.append(
            image.mean().item()
        )

    return float(
        sum(
            values
        )
        / len(
            values
        )
    )


device_intensity_rows = []

for device_name, group in (
    train_df.groupby(
        "device"
    )
):
    device_intensity_rows.append({
        "device":
            device_name,

        "mean_image_intensity":
            mean_intensity_for_paths(
                group[
                    "image_path"
                ]
            )
    })

device_intensity_df = (
    pd.DataFrame(
        device_intensity_rows
    )
)

print(
    device_intensity_df
)


# 60. Harmonization Experiments

Harmonization aims to reduce irrelevant acquisition differences while preserving clinically meaningful information.

Possible approaches include:

- Global training-derived normalization
- Per-image normalization
- Device/site-aware normalization
- Histogram matching
- Learned domain adaptation
- Scanner-invariant representation learning

No method should be assumed beneficial without validation.


# 61. Harmonization Rule 1 — No Leakage

If a harmonization method estimates parameters:

$$
\theta_{harm}
$$

fit them using:

$$
\boxed{
Training\ Data\ Only
}
$$

Then apply the frozen transformation to validation/test.


# 62. Harmonization Baseline A — Global Standardization

This is what our baseline already does:

$$
x'
=
\frac{
x-\mu_{train}
}{
\sigma_{train}
}
$$

One global training distribution is used for all devices.


# 63. Harmonization Baseline B — Per-Image Standardization

Per-image standardization removes image-level mean and scale:

$$
x'
=
\frac{
x-\mu_{image}
}{
\sigma_{image}
}
$$

This can reduce gain differences.

But it may also remove clinically useful absolute intensity information.


In [ ]:
class PerImageStandardize:
    def __call__(
        self,
        tensor
    ):
        mean = tensor.mean()

        std = tensor.std()

        return (
            tensor
            - mean
        ) / (
            std
            + 1e-6
        )


# 64. Per-Image Harmonization Transform


In [ ]:
per_image_train_transform = (
    transforms.Compose([
        ResizeWithPadding(
            128
        ),

        transforms.RandomAffine(
            degrees=7,
            translate=(
                0.04,
                0.04
            )
        ),

        transforms.ToTensor(),

        PerImageStandardize()
    ])
)

per_image_eval_transform = (
    transforms.Compose([
        ResizeWithPadding(
            128
        ),

        transforms.ToTensor(),

        PerImageStandardize()
    ])
)


# 65. Harmonization Baseline C — Training-Derived Device Normalization

If device metadata is known, we can estimate training statistics for each device.

For device $d$:

$$
x'
=
\frac{
x-\mu_{d,train}
}{
\sigma_{d,train}
}
$$

Important:

- Learn statistics from training data only
- Define behavior for unseen devices
- Validate that pathology signal is not removed


In [ ]:
def compute_device_statistics(
    train_dataframe,
    resize_transform
):
    stats = {}

    for device_name, group in (
        train_dataframe.groupby(
            "device"
        )
    ):
        mean, std = (
            compute_grayscale_mean_std(
                group.reset_index(
                    drop=True
                ),
                resize_transform
            )
        )

        stats[
            device_name
        ] = {
            "mean":
                mean,

            "std":
                std
        }

    return stats


device_stats = (
    compute_device_statistics(
        train_df,
        ResizeWithPadding(
            128
        )
    )
)

print(
    json.dumps(
        device_stats,
        indent=2
    )
)


# 66. Device-Normalized Dataset

This dataset chooses normalization statistics based on the row's device.

For unseen devices, we fall back to global training statistics.

This is only one harmonization baseline—not a universal solution.


In [ ]:
class DeviceNormalizedDataset(Dataset):
    def __init__(
        self,
        dataframe,
        device_stats,
        global_mean,
        global_std,
        training=False,
        size=128
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.device_stats = (
            device_stats
        )

        self.global_mean = (
            global_mean
        )

        self.global_std = (
            global_std
        )

        self.training = (
            training
        )

        self.resize = (
            ResizeWithPadding(
                size
            )
        )

        self.to_tensor = (
            transforms.ToTensor()
        )

        self.augment = (
            transforms.RandomAffine(
                degrees=7,
                translate=(
                    0.04,
                    0.04
                )
            )
        )

    def __len__(self):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):
        row = self.dataframe.iloc[
            index
        ]

        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "L"
        )

        image = self.resize(
            image
        )

        if self.training:
            image = self.augment(
                image
            )

        tensor = self.to_tensor(
            image
        )

        device_name = row[
            "device"
        ]

        stats = self.device_stats.get(
            device_name,
            {
                "mean":
                    self.global_mean,

                "std":
                    self.global_std
            }
        )

        tensor = (
            tensor
            - stats[
                "mean"
            ]
        ) / (
            stats[
                "std"
            ]
            + 1e-8
        )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            tensor,
            target,
            index
        )


# 67. Create Harmonized Dataset Variants


In [ ]:
per_image_train_dataset = (
    UltrasoundCSVDataset(
        train_df,
        transform=(
            per_image_train_transform
        )
    )
)

per_image_val_dataset = (
    UltrasoundCSVDataset(
        val_df,
        transform=(
            per_image_eval_transform
        )
    )
)

device_train_dataset = (
    DeviceNormalizedDataset(
        train_df,
        device_stats,
        train_mean,
        train_std,
        training=True
    )
)

device_val_dataset = (
    DeviceNormalizedDataset(
        val_df,
        device_stats,
        train_mean,
        train_std,
        training=False
    )
)


# 68. Harmonization Experiment Design

Compare:

1. Global standardization
2. Per-image standardization
3. Device-specific training-derived normalization

Keep fixed:

- Patient split
- Architecture
- Optimizer
- Epoch count
- Model-selection metric


In [ ]:
def make_simple_loader(
    dataset,
    training
):
    return DataLoader(
        dataset,
        batch_size=24 if training else 48,
        shuffle=training,
        num_workers=0
    )


harmonization_variants = {
    "global":
        (
            baseline_train_dataset,
            baseline_val_dataset
        ),

    "per_image":
        (
            per_image_train_dataset,
            per_image_val_dataset
        ),

    "device_normalized":
        (
            device_train_dataset,
            device_val_dataset
        )
}

print(
    harmonization_variants.keys()
)


# 69. Harmonization Experiment Runner


In [ ]:
def run_harmonization_experiment(
    train_dataset,
    val_dataset,
    seed=42,
    epochs=5
):
    torch.manual_seed(
        seed
    )

    train_loader = (
        make_simple_loader(
            train_dataset,
            training=True
        )
    )

    val_loader = (
        make_simple_loader(
            val_dataset,
            training=False
        )
    )

    model = (
        UltrasoundBaselineCNN(
            num_classes
        )
        .to(
            device
        )
    )

    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )
    )

    history, best_loss = (
        fit_model(
            model,
            train_loader,
            val_loader,
            criterion,
            optimizer,
            device,
            epochs=epochs
        )
    )

    return {
        "model":
            model,

        "history":
            history,

        "best_val_loss":
            best_loss
    }


# 70. Run the Harmonization Comparison

This is an educational baseline comparison.

Real harmonization research should use:

- Multiple seeds
- Grouped cross-validation
- External validation
- Device/site subgroup metrics


In [ ]:
harmonization_results = {}

for name, (
    train_dataset_variant,
    val_dataset_variant
) in harmonization_variants.items():
    print()
    print(
        "=" * 60
    )

    print(
        "Harmonization:",
        name
    )

    harmonization_results[
        name
    ] = (
        run_harmonization_experiment(
            train_dataset_variant,
            val_dataset_variant,
            seed=42,
            epochs=5
        )
    )

print(
    "Harmonization comparison complete."
)


# 71. Compare Harmonization Validation Loss


In [ ]:
for name, result in (
    harmonization_results.items()
):
    print(
        f"{name:20s} | "
        f"best val loss = "
        f"{result['best_val_loss']:.4f}"
    )


# 72. Harmonization Must Be Evaluated Beyond One Overall Metric

Also compare:

- Per-class performance
- Per-site performance
- Per-device performance
- External-site performance
- Calibration
- Explainability/failure cases

A visually cleaner image does not prove better harmonization.


# 73. Transfer-Learning Baseline

A useful real ultrasound project should compare a small CNN from scratch against a pretrained backbone.

We will use:

> **ResNet-18**

The standard pretrained model expects:

$$
3
$$

channels.

Our ultrasound images are grayscale.


# 74. Grayscale-to-RGB Strategy

A simple baseline is:

$$
(1,H,W)
\rightarrow
(3,H,W)
$$

by converting grayscale into three identical channels.

This preserves compatibility with ImageNet-pretrained first-layer weights.


# 75. Pretrained ResNet Transform

For transfer learning:

- Resize/pad
- Convert grayscale to 3 channels
- Mild training augmentation
- Use ImageNet normalization

For real work, compare this against training-derived ultrasound normalization.


In [ ]:
imagenet_mean = [
    0.485,
    0.456,
    0.406
]

imagenet_std = [
    0.229,
    0.224,
    0.225
]

transfer_train_transform = (
    transforms.Compose([
        ResizeWithPadding(
            224
        ),

        transforms.RandomAffine(
            degrees=7,
            translate=(
                0.04,
                0.04
            )
        ),

        transforms.Grayscale(
            num_output_channels=3
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=imagenet_mean,
            std=imagenet_std
        )
    ])
)

transfer_eval_transform = (
    transforms.Compose([
        ResizeWithPadding(
            224
        ),

        transforms.Grayscale(
            num_output_channels=3
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=imagenet_mean,
            std=imagenet_std
        )
    ])
)


# 76. Transfer Datasets


In [ ]:
transfer_train_dataset = (
    UltrasoundCSVDataset(
        train_df,
        transform=(
            transfer_train_transform
        )
    )
)

transfer_val_dataset = (
    UltrasoundCSVDataset(
        val_df,
        transform=(
            transfer_eval_transform
        )
    )
)

transfer_test_dataset = (
    UltrasoundCSVDataset(
        test_df,
        transform=(
            transfer_eval_transform
        )
    )
)


# 77. Load ResNet-18 With Safe Fallback

Pretrained weights may need to be downloaded in Colab.

If download is unavailable, the code falls back to random initialization.

For a genuine transfer-learning experiment, verify:

```python
pretrained_loaded == True
```


In [ ]:
def build_resnet18_transfer_model(
    num_classes
):
    weights = (
        models.ResNet18_Weights.DEFAULT
    )

    try:
        model = models.resnet18(
            weights=weights
        )

        pretrained_loaded = True

    except Exception as error:
        print(
            "Pretrained weights unavailable."
        )

        print(
            "Reason:",
            error
        )

        model = models.resnet18(
            weights=None
        )

        pretrained_loaded = False

    for parameter in (
        model.parameters()
    ):
        parameter.requires_grad = False

    in_features = (
        model.fc.in_features
    )

    model.fc = nn.Linear(
        in_features,
        num_classes
    )

    return (
        model,
        pretrained_loaded
    )


transfer_model, pretrained_loaded = (
    build_resnet18_transfer_model(
        num_classes
    )
)

print(
    "Pretrained loaded:",
    pretrained_loaded
)


# 78. Train Only the Classifier Head


In [ ]:
transfer_train_loader = DataLoader(
    transfer_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

transfer_val_loader = DataLoader(
    transfer_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

transfer_model = transfer_model.to(
    device
)

transfer_criterion = (
    nn.CrossEntropyLoss()
)

transfer_optimizer = (
    torch.optim.AdamW(
        filter(
            lambda p:
                p.requires_grad,
            transfer_model.parameters()
        ),
        lr=1e-3,
        weight_decay=1e-4
    )
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in transfer_model.parameters()
        if p.requires_grad
    )
)


# 79. Feature-Extraction Training

This code is the same PyTorch training loop.

Only the trainable parameter set changed.


In [ ]:
transfer_history, transfer_best_loss = (
    fit_model(
        transfer_model,
        transfer_train_loader,
        transfer_val_loader,
        transfer_criterion,
        transfer_optimizer,
        device,
        epochs=5
    )
)

print(
    "Transfer best val loss:",
    transfer_best_loss
)


# 80. Fine-Tuning the Final ResNet Stage

After training the head, we can unfreeze:

```python
model.layer4
```

and use a smaller learning rate for pretrained weights.


In [ ]:
for parameter in (
    transfer_model
    .layer4
    .parameters()
):
    parameter.requires_grad = True

fine_tune_optimizer = (
    torch.optim.AdamW([
        {
            "params":
                transfer_model
                .layer4
                .parameters(),

            "lr":
                1e-4
        },
        {
            "params":
                transfer_model
                .fc
                .parameters(),

            "lr":
                1e-3
        }
    ],
    weight_decay=1e-4)
)

print(
    "Fine-tuning parameter groups:",
    len(
        fine_tune_optimizer
        .param_groups
    )
)


# 81. Fine-Tune Carefully

For a real project:

- Use validation data
- Keep the best checkpoint
- Consider lower backbone learning rate
- Monitor overfitting
- Do not choose the strategy from test performance


# 82. Patient-Level Prediction Aggregation

Suppose one patient has multiple images:

$$
p_1,\ p_2,\ p_3
$$

A patient-level prediction can average probabilities:

$$
\boxed{
\bar{p}
=
\frac{
1
}{
K
}
\sum_{k=1}^{K}
p_k
}
$$

Then choose:

$$
argmax(\bar{p})
$$


In [ ]:
def aggregate_to_patient_level(
    prediction_dataframe,
    num_classes
):
    probability_columns = [
        f"prob_class_{index}"
        for index in range(
            num_classes
        )
    ]

    patient_rows = []

    for patient_id, group in (
        prediction_dataframe.groupby(
            "patient_id"
        )
    ):
        mean_probabilities = (
            group[
                probability_columns
            ]
            .mean()
            .to_numpy()
        )

        predicted_label = int(
            mean_probabilities.argmax()
        )

        true_labels = (
            group[
                "true_label"
            ]
            .unique()
        )

        assert len(
            true_labels
        ) == 1

        patient_rows.append({
            "patient_id":
                patient_id,

            "true_label":
                int(
                    true_labels[
                        0
                    ]
                ),

            "predicted_label":
                predicted_label,

            **{
                column:
                    float(
                        mean_probabilities[
                            index
                        ]
                    )
                for index, column
                in enumerate(
                    probability_columns
                )
            }
        })

    return pd.DataFrame(
        patient_rows
    )


# 83. Patient-Level Validation Metrics


In [ ]:
patient_val_predictions = (
    aggregate_to_patient_level(
        val_predictions_df,
        num_classes
    )
)

patient_val_accuracy = (
    patient_val_predictions[
        "true_label"
    ].to_numpy()
    ==
    patient_val_predictions[
        "predicted_label"
    ].to_numpy()
).mean()

print(
    "Patient-level validation accuracy:",
    patient_val_accuracy
)


# 84. Why Image-Level and Patient-Level Metrics Differ

One patient may contribute many correlated frames.

Image-level evaluation gives that patient more weight.

Patient-level aggregation gives each patient one final prediction.

Report the evaluation unit clearly.


# 85. Save Reproducible Experiment Outputs

Create an experiment directory containing:

- Config
- Split manifest
- Checkpoint
- Validation predictions
- Test predictions
- Metrics
- Device/site tables


In [ ]:
EXPERIMENT_DIR = (
    OUTPUT_DIR
    / "baseline_experiment"
)

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    EXPERIMENT_DIR
)


# 86. Save Experiment Configuration


In [ ]:
experiment_config = {
    "experiment_name":
        "baseline_experiment",

    "seed":
        42,

    "input_size":
        128,

    "num_classes":
        num_classes,

    "split_unit":
        "patient",

    "model":
        "UltrasoundBaselineCNN",

    "normalization":
        "global_training_mean_std",

    "train_mean":
        float(
            train_mean
        ),

    "train_std":
        float(
            train_std
        ),

    "selected_imbalance_strategy":
        selected_baseline[
            "name"
        ],

    "checkpoint_metric":
        "validation_loss"
}

(
    EXPERIMENT_DIR
    / "config.json"
).write_text(
    json.dumps(
        experiment_config,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Config saved."
)


# 87. Save Validation Predictions


In [ ]:
val_predictions_df.to_csv(
    EXPERIMENT_DIR
    / "validation_predictions.csv",
    index=False
)

site_metrics.to_csv(
    EXPERIMENT_DIR
    / "validation_site_metrics.csv",
    index=False
)

device_metrics.to_csv(
    EXPERIMENT_DIR
    / "validation_device_metrics.csv",
    index=False
)

print(
    "Validation outputs saved."
)


# 88. Save the Selected Baseline Model


In [ ]:
torch.save(
    selected_baseline[
        "model"
    ].state_dict(),
    EXPERIMENT_DIR
    / "best_model_weights.pth"
)

print(
    "Model weights saved."
)


# 89. Final Test Evaluation

Only after:

- Model selection
- Imbalance strategy selection
- Harmonization choice
- Hyperparameter choice

should we evaluate the final test set.


In [ ]:
selected_test_loss, selected_test_accuracy = (
    evaluate_one_epoch(
        selected_baseline[
            "model"
        ],
        baseline_test_loader,
        nn.CrossEntropyLoss(),
        device
    )
)

test_predictions_df = (
    collect_predictions(
        selected_baseline[
            "model"
        ],
        baseline_test_dataset,
        baseline_test_loader,
        device
    )
)

print(
    "Test loss:",
    selected_test_loss
)

print(
    "Test accuracy:",
    selected_test_accuracy
)


# 90. Patient-Level Test Evaluation


In [ ]:
patient_test_predictions = (
    aggregate_to_patient_level(
        test_predictions_df,
        num_classes
    )
)

patient_test_accuracy = (
    patient_test_predictions[
        "true_label"
    ].to_numpy()
    ==
    patient_test_predictions[
        "predicted_label"
    ].to_numpy()
).mean()

print(
    "Patient-level test accuracy:",
    patient_test_accuracy
)


# 91. Test Site/Device Metrics


In [ ]:
test_site_metrics = (
    subgroup_accuracy(
        test_predictions_df,
        "site"
    )
)

test_device_metrics = (
    subgroup_accuracy(
        test_predictions_df,
        "device"
    )
)

print(
    test_site_metrics
)

print()

print(
    test_device_metrics
)


# 92. Save Final Test Outputs


In [ ]:
test_predictions_df.to_csv(
    EXPERIMENT_DIR
    / "test_predictions.csv",
    index=False
)

patient_test_predictions.to_csv(
    EXPERIMENT_DIR
    / "patient_test_predictions.csv",
    index=False
)

test_site_metrics.to_csv(
    EXPERIMENT_DIR
    / "test_site_metrics.csv",
    index=False
)

test_device_metrics.to_csv(
    EXPERIMENT_DIR
    / "test_device_metrics.csv",
    index=False
)

print(
    "Test outputs saved."
)


# 93. Save Final Metrics


In [ ]:
final_metrics = {
    "image_level_test_loss":
        float(
            selected_test_loss
        ),

    "image_level_test_accuracy":
        float(
            selected_test_accuracy
        ),

    "patient_level_test_accuracy":
        float(
            patient_test_accuracy
        )
}

(
    EXPERIMENT_DIR
    / "final_metrics.json"
).write_text(
    json.dumps(
        final_metrics,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        final_metrics,
        indent=2
    )
)


# 94. Recommended Real Ultrasound Project Structure

```text
ultrasound_project/
│
├── data/
│   ├── raw/
│   └── processed/
│
├── metadata/
│   └── images.csv
│
├── splits/
│   ├── patient_split.json
│   └── image_manifest.csv
│
├── src/
│   ├── datasets.py
│   ├── transforms.py
│   ├── harmonization.py
│   ├── models.py
│   ├── train.py
│   ├── evaluate.py
│   └── explain.py
│
├── configs/
│
├── experiments/
│   └── experiment_001/
│       ├── config.json
│       ├── model.pth
│       ├── validation_predictions.csv
│       ├── test_predictions.csv
│       └── metrics.json
│
└── README.md
```


# 95. Preparing a Research-Quality Metadata CSV

Before modeling, verify:

1. Every image path is valid
2. Every label is defined
3. Patient IDs are stable
4. Study IDs are stable
5. Site information is correct
6. Device/scanner information is available if relevant
7. Duplicate images are identified
8. Multiple images from one patient are recognized


# 96. Duplicate and Near-Duplicate Checks

Possible duplicate sources:

- Same file copied twice
- Same frame exported twice
- Same study image stored under different names
- Augmented images generated before splitting

Exact duplicates can be checked with file hashes.

Near-duplicates may require image similarity methods.

Do this before creating final splits.


# 97. Site Leakage

Suppose:

$$
Site_A
$$

contains mostly class 0,

while:

$$
Site_B
$$

contains mostly class 1.

The model may learn site style instead of pathology.

Always inspect:

```python
pd.crosstab(
    metadata["site"],
    metadata["label"]
)
```


In [ ]:
print(
    pd.crosstab(
        metadata[
            "site"
        ],
        metadata[
            "label"
        ]
    )
)


# 98. Device Leakage

Also inspect:

$$
Device
\times
Label
$$

association.


In [ ]:
print(
    pd.crosstab(
        metadata[
            "device"
        ],
        metadata[
            "label"
        ]
    )
)


# 99. Harmonization Should Not Remove Disease Signal

A harmonization method can overcorrect.

For example, if clinically relevant echogenicity differs between classes, aggressive per-image normalization may remove part of that signal.

Therefore compare:

- Performance
- Calibration
- Site/device robustness
- Error types
- Explanation maps

after harmonization.


# 100. Unseen Device Problem

Suppose validation contains:

`Device_D`

but training contains only:

- Device_A
- Device_B
- Device_C

A device-specific normalizer has no learned statistics for `Device_D`.

You must define a fallback strategy such as:

- Global training normalization
- Device-independent harmonization
- External calibration study

This is an important deployment question.


# 101. External-Site Evaluation

A strong medical-imaging study may include:

$$
\boxed{
Internal\ Validation
}
$$

and:

$$
\boxed{
External\ Site\ Test
}
$$

An external site provides stronger evidence that the model generalizes beyond one acquisition environment.


# 102. Temporal Split

Sometimes a realistic evaluation uses:

- Earlier patients for development
- Later patients for testing

This tests generalization across time.

Useful when clinical workflow or scanners change over time.


# 103. Study-Level vs Patient-Level Split

If one patient has multiple studies:

- Patient-level split prevents all cross-study leakage
- Study-level split may still put the same patient in multiple sets

Choose the split unit based on the intended scientific question.


# 104. View-Specific Models

Ultrasound datasets may contain multiple views.

Possible strategies:

- One model for all views
- Separate model per view
- Include view metadata
- Multi-view aggregation

Do not mix view distributions blindly.


# 105. Patient-Level Aggregation Strategies

Possible aggregation methods include:

$$
\begin{array}{|c|c|}
\hline
Mean\ probability & \text{Average evidence} \\
\hline
Max\ probability & \text{Any strong positive image} \\
\hline
Majority\ vote & \text{Most common predicted class} \\
\hline
Learned\ aggregation & \text{Train a study/patient model} \\
\hline
\end{array}
$$

The aggregation rule should be defined before final testing.


# 106. Calibration for Clinical Probabilities

If you want to interpret:

$$
P(disease)=0.80
$$

as a meaningful risk estimate, evaluate calibration.

A model can have high AUROC and still be poorly calibrated.

For clinical decision support, consider:

- Reliability diagrams
- Calibration error
- Post-hoc calibration fit on validation data


# 107. Threshold Selection

For binary ultrasound classification, the final threshold might be chosen to:

- Maximize F1
- Achieve minimum sensitivity
- Achieve minimum specificity
- Minimize clinical cost

Choose on validation data.

Freeze before final test evaluation.


# 108. Repeated Experiments

A single random initialization can be lucky or unlucky.

For research-quality comparison:

- Repeat with multiple seeds
- Use patient-grouped cross-validation
- Report mean and variability

This becomes especially important with small medical datasets.


# 109. Research-Quality Harmonization Comparison

A stronger harmonization experiment should compare:

1. No harmonization beyond basic scaling
2. Global training standardization
3. Per-image normalization
4. Device/site-aware normalization
5. More advanced harmonization method

For every method, keep the downstream model/evaluation protocol fixed.


# 110. What Should Be Frozen Before Final Testing?

Before opening the final test set, freeze:

- Patient split
- Preprocessing
- Harmonization method
- Architecture
- Hyperparameters
- Class-imbalance strategy
- Checkpoint-selection metric
- Threshold/aggregation rule


# 111. Reproducibility Checklist

Save:

1. Metadata CSV
2. Split manifest
3. Class mapping
4. Preprocessing config
5. Harmonization config
6. Random seed
7. Model checkpoint
8. Validation predictions
9. Test predictions
10. Subgroup metrics
11. Software versions


# 112. Practical Debugging Checklist

Before a long real-data run:

1. Open one file manually
2. Inspect original size
3. Inspect transformed size
4. Check tensor range
5. Check mean/std
6. Inspect one training batch
7. Verify class counts
8. Verify patient split
9. Verify site/device distributions
10. Run one forward pass
11. Run one backward pass
12. Overfit a tiny batch


# 113. Common Mistake — Paths Are Relative to the Wrong Folder

CSV paths may be:

- Absolute
- Relative to project root
- Relative to CSV location

Define one consistent path convention.

For real projects, it is often helpful to store relative paths and resolve them against a known project root.


# 114. Common Mistake — Reading Ultrasound as RGB Accidentally

Some grayscale PNG/JPEG files are stored with three channels.

If your baseline CNN expects one channel, convert explicitly:

```python
Image.open(path).convert("L")
```


# 115. Common Mistake — Normalizing Before Splitting

If normalization statistics are calculated before the split, validation/test data influence preprocessing.

Split first.

Fit preprocessing second.


# 116. Common Mistake — Using Random Augmentation in Validation

Validation/test should usually be deterministic.

Otherwise metrics fluctuate because the data itself changes each evaluation.


# 117. Common Mistake — Oversampling Validation or Test Data

Weighted sampling is a training strategy.

Do not change the natural validation/test distribution to make metrics look balanced.


# 118. Common Mistake — Using Both Strong Sampling and Strong Class Weights Automatically

This may overcorrect the imbalance.

Compare:

- No weighting
- Weighted loss
- Weighted sampling

systematically.


# 119. Common Mistake — Harmonizing Using Validation/Test Statistics

That is leakage.

All learned harmonization parameters must come from training data only.


# 120. Common Mistake — Reporting Only Overall Performance

Report relevant:

- Per-class metrics
- Site metrics
- Device metrics
- Patient-level metrics
- External-site performance

Overall accuracy can hide important failure modes.


# 121. Common Mistake — Choosing Harmonization by Visual Appearance

Cleaner-looking ultrasound does not guarantee better machine-learning performance.

Evaluate objectively.


# 122. Common Mistake — Tuning Transfer Learning on the Test Set

Use validation data to decide:

- Freeze vs fine-tune
- Learning rates
- Number of unfrozen layers
- Input normalization

Then evaluate test once.


# 123. Practice Exercises

## Exercise 1

Create a metadata CSV with:

- `image_path`
- `label`
- `patient_id`
- `site`
- `device`

## Exercise 2

Create patient-level train/validation/test splits.

## Exercise 3

Save the split manifest to JSON.

## Exercise 4

Write a file-path based grayscale image `Dataset`.

## Exercise 5

Implement aspect-ratio preserving resize + padding.

## Exercise 6

Compute training-only grayscale mean and standard deviation.

## Exercise 7

Calculate class weights and create weighted `CrossEntropyLoss`.

## Exercise 8

Create a `WeightedRandomSampler`.

## Exercise 9

Build a one-channel CNN with adaptive average pooling.

## Exercise 10

Aggregate image-level probabilities into one patient-level prediction.


# 124. Conceptual Challenges

## Challenge 1

Why is a metadata CSV more useful than only storing images in class folders?

## Challenge 2

Why should patient IDs be preserved throughout the pipeline?

## Challenge 3

Why can direct resizing distort ultrasound anatomy?

## Challenge 4

Why must normalization statistics come from training only?

## Challenge 5

What is the difference between weighted loss and weighted sampling?

## Challenge 6

Why should validation/test distributions not be oversampled?

## Challenge 7

Why can device-specific intensity differences become shortcuts?

## Challenge 8

Why might per-image standardization help harmonization?

## Challenge 9

Why might per-image standardization hurt?

## Challenge 10

Why must harmonization parameters be learned only on training data?

## Challenge 11

Why should site/device subgroup metrics be reported?

## Challenge 12

Why can image-level and patient-level results differ?

## Challenge 13

Why does a pretrained RGB model require special handling for grayscale ultrasound?

## Challenge 14

Why should an external-site test be valuable?

## Challenge 15

Why should all development decisions be frozen before final test evaluation?


# 125. Exercise Solutions


In [ ]:
# Exercise 2 and 3
exercise_patient_table = pd.DataFrame({
    "patient_id": [
        f"P{i:02d}"
        for i in range(
            30
        )
    ],

    "label": [
        i % 3
        for i in range(
            30
        )
    ]
})

exercise_train_ids, exercise_val_ids, exercise_test_ids = (
    stratified_patient_split(
        exercise_patient_table,
        seed=7
    )
)

assert exercise_train_ids.isdisjoint(
    exercise_val_ids
)

assert exercise_train_ids.isdisjoint(
    exercise_test_ids
)

assert exercise_val_ids.isdisjoint(
    exercise_test_ids
)

exercise_manifest = {
    "train":
        sorted(
            exercise_train_ids
        ),

    "val":
        sorted(
            exercise_val_ids
        ),

    "test":
        sorted(
            exercise_test_ids
        )
}

Path(
    "exercise_split_manifest.json"
).write_text(
    json.dumps(
        exercise_manifest,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Exercises 2-3 complete."
)


In [ ]:
# Exercise 4
class ExerciseUltrasoundDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = (
            transform
        )

    def __len__(self):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):
        row = self.dataframe.iloc[
            index
        ]

        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "L"
        )

        if self.transform is not None:
            image = self.transform(
                image
            )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            target,
            index
        )


In [ ]:
# Exercise 5
exercise_resize = ResizeWithPadding(
    128
)

exercise_image = Image.open(
    metadata.iloc[
        0
    ][
        "image_path"
    ]
).convert(
    "L"
)

exercise_resized = exercise_resize(
    exercise_image
)

print(
    "Exercise 5:",
    exercise_resized.size
)


In [ ]:
# Exercise 6
exercise_mean, exercise_std = (
    compute_grayscale_mean_std(
        train_df,
        ResizeWithPadding(
            128
        )
    )
)

print(
    "Exercise 6:",
    exercise_mean,
    exercise_std
)


In [ ]:
# Exercise 7 and 8
exercise_counts = (
    train_df[
        "label"
    ]
    .value_counts()
    .sort_index()
)

exercise_counts_tensor = torch.tensor([
    exercise_counts.get(
        class_index,
        0
    )
    for class_index in range(
        num_classes
    )
],
dtype=torch.float32)

exercise_class_weights = (
    exercise_counts_tensor.sum()
    / (
        num_classes
        * exercise_counts_tensor.clamp_min(
            1
        )
    )
)

exercise_loss = (
    nn.CrossEntropyLoss(
        weight=exercise_class_weights
    )
)

exercise_sample_weights = torch.tensor([
    exercise_class_weights[
        int(label)
    ].item()
    for label in train_df[
        "label"
    ]
],
dtype=torch.double)

exercise_sampler = (
    WeightedRandomSampler(
        exercise_sample_weights,
        num_samples=len(
            exercise_sample_weights
        ),
        replacement=True
    )
)

print(
    "Exercises 7-8 complete."
)


In [ ]:
# Exercise 9
exercise_cnn = nn.Sequential(
    nn.Conv2d(
        1,
        16,
        3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        16,
        32,
        3,
        padding=1
    ),
    nn.ReLU(),

    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(
        32,
        num_classes
    )
)

print(
    exercise_cnn
)


In [ ]:
# Exercise 10
exercise_prediction_df = pd.DataFrame({
    "patient_id": [
        "P1",
        "P1",
        "P2",
        "P2"
    ],

    "true_label": [
        0,
        0,
        1,
        1
    ],

    "predicted_label": [
        0,
        0,
        1,
        2
    ],

    "prob_class_0": [
        0.8,
        0.7,
        0.1,
        0.2
    ],

    "prob_class_1": [
        0.1,
        0.2,
        0.6,
        0.4
    ],

    "prob_class_2": [
        0.1,
        0.1,
        0.3,
        0.4
    ]
})

exercise_patient_predictions = (
    aggregate_to_patient_level(
        exercise_prediction_df,
        num_classes=3
    )
)

print(
    exercise_patient_predictions
)


# 126. Conceptual Challenge Solutions

## Challenge 1

A metadata CSV preserves information such as patient, study, site, device, and view. Class folders alone usually cannot represent all of those research variables cleanly.

## Challenge 2

Patient IDs are needed to prevent patient leakage and to calculate patient-level metrics.

## Challenge 3

Direct resizing can stretch one spatial axis more than the other and alter geometric relationships. Aspect-ratio preserving resize plus padding avoids that distortion.

## Challenge 4

Validation/test-derived normalization statistics leak evaluation information into preprocessing.

## Challenge 5

Weighted loss changes the importance of each class in the objective. Weighted sampling changes how often training samples are seen.

## Challenge 6

Validation/test sets should reflect the natural evaluation distribution. Oversampling them changes the distribution and can make metrics misleading.

## Challenge 7

Device-specific gain, contrast, post-processing, and noise can correlate with labels and become easier shortcuts than pathology.

## Challenge 8

Per-image standardization can reduce image-to-image gain and brightness differences.

## Challenge 9

It can remove useful absolute intensity information that may genuinely relate to pathology.

## Challenge 10

Otherwise the harmonization transformation itself has learned from validation/test data.

## Challenge 11

Overall performance can hide failure on one hospital, scanner, or acquisition setting.

## Challenge 12

Patients with many images have greater influence on image-level metrics. Patient-level aggregation gives each patient one final decision.

## Challenge 13

ImageNet-pretrained CNNs normally expect three channels and ImageNet-like preprocessing, while ultrasound is often one-channel grayscale.

## Challenge 14

External-site testing provides stronger evidence of robustness to scanner, operator, and acquisition-domain shifts.

## Challenge 15

Using the final test set to choose preprocessing, harmonization, model, threshold, or aggregation rule turns it into another validation set.


# 127. Key Takeaways

In this notebook, we built a realistic ultrasound classification workflow using:

- Metadata CSV files
- File paths
- Patient IDs
- Study IDs
- Sites
- Devices
- Patient-level split manifests
- Grayscale image loading
- Variable image-size handling
- Aspect-ratio preserving resize and padding
- Training-only normalization
- Training-only augmentation
- Custom file-based `Dataset`
- Class imbalance analysis
- Weighted losses
- Weighted sampling
- Baseline ultrasound CNN
- Validation checkpointing
- Site-aware evaluation
- Device-aware evaluation
- Harmonization baselines
- Training-only device normalization
- Transfer-learning baseline
- Grayscale-to-RGB adaptation
- Patient-level probability aggregation
- Reproducible experiment output folders
- Final test evaluation

The most important split rule is:

$$
\boxed{
Patient\ Independence
}
$$

The most important preprocessing rule is:

$$
\boxed{
Fit\ Preprocessing/Harmonization\ on\ Training\ Only
}
$$

The most important multi-site rule is:

$$
\boxed{
Evaluate\ by\ Site\ and\ Device
}
$$

And the most important research principle is:

$$
\boxed{
Do\ Not\ Optimize\ Against\ the\ Final\ Test\ Set
}
$$


# 128. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why use a metadata CSV?
2. Why preserve patient IDs?
3. Why preserve study IDs?
4. Why preserve site/device information?
5. Why verify file paths before training?
6. Why split patients rather than images?
7. Why save split manifests?
8. Why inspect split class distributions?
9. Why inspect site/device distributions?
10. Why explicitly convert ultrasound to grayscale?
11. Why can variable image sizes break batching?
12. What is the advantage of resize + padding?
13. Why compute mean/std from training only?
14. Why should augmentation be training-only?
15. What does weighted loss change?
16. What does weighted sampling change?
17. Why should you not oversample validation/test?
18. What is the baseline CNN input shape?
19. Why use adaptive average pooling?
20. Why evaluate by site?
21. Why evaluate by device?
22. What is harmonization?
23. Why can per-image normalization help?
24. Why can per-image normalization hurt?
25. Why must harmonization be fit on training only?
26. How can an RGB-pretrained model accept grayscale ultrasound?
27. What is feature extraction?
28. What is fine-tuning?
29. Why can patient-level aggregation be needed?
30. What should be saved in a reproducible experiment folder?


# Next Notebook

# 25 — Research-Grade Experiments, Ablations, and Patient-Grouped Cross-Validation

In the next notebook, we will study:

- Why one train/validation split may be unstable
- Patient-grouped cross-validation
- Stratified group folds
- Repeated experiments with multiple seeds
- Hyperparameter tuning without leakage
- Nested-validation intuition
- Ablation studies
- Comparing preprocessing strategies fairly
- Comparing harmonization methods fairly
- Confidence intervals
- Patient-level bootstrap
- External validation
- Statistical model comparison intuition
- Experiment tracking
- Building reproducible result tables
- Reporting research-quality deep-learning experiments
